# Analysis, Plots and *Insights*

This *notebook* holds the charts and the *insights* drawn from the
*Dataframe* cleaned in the previous one.

<table align="left">
  <tr>
    <td>
      <a href="https://colab.research.google.com/github/ValentimPiazera/EDA-Carros-Eletricos-Washington/blob/main/notebooks/II-analysis.ipynb" target="_parent">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
      </a>
    </td>
    <td>
      <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ValentimPiazera/EDA-Carros-Eletricos-Washington/blob/main/notebooks/II-analysis.ipynb">
        <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/>
      </a>
    </td>
  </tr>
</table>

> **Note:** this notebook is written to run from the **repository root**, not from `notebooks/`.
> Locally, `.vscode/settings.json` already points VS Code's Jupyter root at the workspace folder;
> from a terminal, launch Jupyter from the repository root.
>
> If you're running on Google Colab or Kaggle, run the cell below first: it clones the repository,
> which is what brings in `data/processed/` — the file exported by `I-cleaning`.
>
> On **Kaggle**, make sure Internet access is enabled: *Settings → Internet → On*.

In [1]:
# Setup: clone repo when running on Colab/Kaggle (skip if running locally!)
import os

REPO = "EDA-Carros-Eletricos-Washington"

IN_COLAB_OR_KAGGLE = (
    "google.colab" in str(get_ipython())
    or "kaggle" in os.environ.get("KAGGLE_URL_BASE", "").lower()
)

# The second test is what makes the cell re-runnable: once the working
# directory is the repository root there is nothing left to clone or enter.
if IN_COLAB_OR_KAGGLE and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        !git clone https://github.com/ValentimPiazera/{REPO}.git
    # The repo root becomes the working directory, so the `data/` paths
    # and `from src import cleaning` resolve as they do locally.
    %cd {REPO}


## Part I — Loading and Imports

Much like the section of the same name in the previous *notebook*, except
that this time I am only checking that everything done there worked, since I
already know the data.

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src import utils, viz

In [3]:
df = pd.read_csv("data/processed/ev_population_washington_clean.csv")
df.head(10)

,County,City,Model Year,Make,Model,Electric Vehicle Type,CAFV Status,Electric Range,Electric Utility,Vehicle Age
0,Kitsap,Bainbridge Island,2018,TESLA,MODEL 3,BEV,Eligible,215.0,Puget Sound Energy,8
1,Kitsap,Port Orchard,2011,NISSAN,LEAF,BEV,Eligible,73.0,Puget Sound Energy,15
2,King,Bothell,2011,NISSAN,LEAF,BEV,Eligible,73.0,Puget Sound Energy,15
3,Yakima,Selah,2020,CHEVROLET,BOLT EV,BEV,Eligible,259.0,Pacificorp,6
4,King,Seattle,2020,KIA,NIRO,PHEV,Not Eligible (Low Range),26.0,City Of Seattle,6
5,Thurston,Olympia,2023,KIA,NIRO,PHEV,Eligible,33.0,Puget Sound Energy,3
6,King,Seattle,2016,AUDI,A3,PHEV,Not Eligible (Low Range),16.0,City Of Seattle,10
7,Thurston,Tumwater,2018,TESLA,MODEL 3,BEV,Eligible,215.0,Puget Sound Energy,8
8,Kitsap,Bremerton,2018,TESLA,MODEL S,BEV,Eligible,249.0,Puget Sound Energy,8
9,Snohomish,Lynnwood,2017,CHEVROLET,VOLT,PHEV,Eligible,53.0,Puget Sound Energy,9


## Part II — Fleet Overview

This part holds charts with general information about the fleet, based on all
270 thousand rows of the *dataframe*, among them:

* **1 —** The top 10 makes in the fleet: which one has the most vehicles
registered?

* **2 —** BEV versus PHEV: which one dominates the fleet?

* **3 —** How is the fleet distributed by age?

* **4 —** How do the vehicles eligible for *CAFV* compare with those that are
not?

* **5 —** And finally, which cities hold the largest concentration of
vehicles?

In [4]:
top_10_makes = utils.top_makes(df)

top_10_makes

,Make,Registrations
0,TESLA,110210
1,CHEVROLET,19015
2,NISSAN,15938
3,FORD,14908
4,KIA,13600
5,TOYOTA,11350
6,BMW,11176
7,HYUNDAI,9806
8,RIVIAN,8491
9,VOLKSWAGEN,7356


In [5]:
fig = viz.top_makes_bar(top_10_makes)

fig.show()

### Figure 1 — Analysis

Simple as this statistic is, it says a great deal about the state of the
fleet, and even raises questions about the market. Such as: given how recent
its tradition in the North American market is, why does Tesla hold such a wide
lead? Rivian, which "appeared" in the American market recently, seems to be
growing fast and is interesting to watch.

In [6]:
bev_vs_phev = utils.vehicle_type_counts(df)

bev_vs_phev

,Type,Registrations
0,BEV,215829
1,PHEV,54625


In [7]:
fig = viz.vehicle_type_pie(bev_vs_phev)

fig.show()

### Figure 2 — Analysis

The picture speaks for itself: BEVs dominate the market, and a quick search on
any engine, or even on the Washington State government site, shows why. First,
BEVs are more affordable and exempt from certain taxes; second, being 100%
electric their mechanics are simpler, with no extra combustion engine or extra
parts, which brings the price down and makes maintenance easier — even though
each of the two has its own specific advantages.

In [8]:
df["Vehicle Age"].mean().round()

np.float64(4.0)

In [9]:
by_model_year = utils.registrations_by_year(df)

fig = viz.model_year_area(by_model_year)

fig.show()

### Figure 3 — Analysis

Two readings are possible here. Either people are adopting EVs with a
definitive boom from 2020 on, or people change cars often, in which case we
cannot tell whether their previous vehicle was already electric. The truth is
a little of both: the state offered incentives and tax exemptions to electric
cars shortly before the definitive boom, purchasing power in the country is
high, and getting around an American city by anything other than a car is
difficult.

In [10]:
cafv_counts = utils.cafv_by_type(df)

cafv_counts

,Electric Vehicle Type,CAFV Status,Registrations
0,BEV,Eligible,45391
1,PHEV,Eligible,30625
2,PHEV,Not Eligible (Low Range),23961


In [11]:
fig = viz.cafv_by_type_bar(cafv_counts)

fig.show()

### Figure 4 — Analysis

The difference between the two types is striking, but there is an explanation
for it: *CAFV* is a measure of whether a car counts as *clean*, and vehicles
with at least **30 miles** (roughly 48 km) qualify. That follows from the
nature of the vehicles — a BEV is obviously expected to go much further on the
battery alone.

*CAFV* is also a "ticket" to exemption from certain taxes.

In [12]:
top_15_cities = utils.top_cities(df)

top_15_cities.head(15)

,City,Registrations
0,Everett,4266
1,Lynnwood,4425
2,Bellingham,4507
3,Spokane,4542
4,Kent,4622
5,Tacoma,5900
6,Olympia,6366
7,Renton,7448
8,Sammamish,7539
9,Kirkland,7702


In [13]:
fig = viz.top_cities_funnel(top_15_cities)

fig.show()

### Figure 5 — Analysis

As expected, *Seattle* holds almost 4 times as many cars as the runner-up;
there is also a certain evenness between the groups, in that once *Seattle* is
set aside the bars climb gently and gradually.

**Important:** *Washington DC* is not in *Washington*. It is a federal
district on the other side of the country, similar to the Distrito Federal in
Brazil.

## Part III — Range and Technological Evolution

I set this part aside for a general view of how the technology evolved. I
decided on a set of *scatter plots*, grouping the vehicles by period and
looking at volume against range. The *scatter plot* is perfect for it: a
cluster of dots together points to an average, while a lot of range on few
units tends to be a luxury car that few can afford, and vice versa.

I chose 2 charts over two specific historical windows: 2015 – 2019, the first
boom visible in figure 3, and 2020 to 2026, the definitive one — which also
makes it possible to compare how range technology evolved between them.

In [14]:
models_2015_2019 = utils.models_by_period(df, 2015, 2019)

models_2015_2019

,Model,Mean Electric Range,Registrations
0,AUDI A3 PHEV,16.0,532
1,AUDI E-TRON BEV,204.0,529
2,BMW 330E PHEV,14.0,179
3,BMW 530E PHEV,15.0,288
4,BMW 740E PHEV,14.0,26
5,BMW I3 BEV,104.0,387
6,BMW I3 PHEV,90.0,961
7,BMW I8 PHEV,15.0,87
8,BMW X5 PHEV,14.0,465
9,CADILLAC CT6 PHEV,31.0,13


In [15]:
fig = viz.early_period_scatter(models_2015_2019)

fig.show()

### Figure 6 — Analysis

We see a fairly diverse market, with countless PHEVs in the bottom left corner
and BEV models from Tesla, Chevrolet, Hyundai and Nissan near the top of the
range axis — but again Tesla dominates, placing many models at the top centre
and the top right. Small dots also show up in the top left corner, which are
BEVs from luxury makes such as Jaguar and Audi.

In [16]:
models_2020_2026 = utils.models_by_period(df, 2020, 2026)

models_2020_2026

,Model,Mean Electric Range,Registrations
1,ALFA ROMEO TONALE PHEV,33.0,98
3,AUDI A7 E PHEV,24.0,11
4,AUDI A8 E PHEV,17.0,4
5,AUDI E-TRON BEV,222.0,678
7,AUDI E-TRON SPORTBACK BEV,218.0,208
...,...,...,...
153,VOLVO S60 PHEV,33.0,196
154,VOLVO S90 PHEV,35.0,17
155,VOLVO V60 PHEV,39.0,100
157,VOLVO XC60 PHEV,30.0,1658


In [17]:
fig = viz.late_period_scatter(models_2020_2026)

fig.show()

### Figure 7 — Analysis

Teslas are the kings of range and have tens of thousands of units on the road.
The average for PHEVs in particular sits between 30 and 50 miles, judging by
how many dots cluster in the bottom left corner.

Only luxury models such as Polestar and Porsche stand up to Tesla on range,
but their small dot, well over to the left, says there are few of them in
circulation.

## Part IV — Market Share

In figure 1, and later in 6 and 7, I noticed that Tesla dominates the fleet by
a wide margin in sheer numbers, whether in a specific window as in charts 6
and 7 or across the whole fleet as in chart 1. But how large is its slice of
the *market share* in numbers? That is what I set this last plotting part
aside to express as a percentage.

In [18]:
share_by_make = utils.market_share(df)

share_by_make

,Make,Share
0,TESLA,40.7
1,CHEVROLET,7.0
2,NISSAN,5.9
3,FORD,5.5
4,KIA,5.0
5,TOYOTA,4.2
6,BMW,4.1
7,HYUNDAI,3.6
8,RIVIAN,3.1
9,VOLKSWAGEN,2.7


In [19]:
fig = viz.market_share_treemap(share_by_make)

fig.show()

### Figure 8 — Analysis

As the earlier charts already pointed out, Tesla dominates the market even
when every vehicle in the fleet is taken into account, as above. Other makes
hover between 7 and 4 per cent, fighting to be the best of the rest, and after
that come the minute shares of niche and luxury brands.

## Final Verdict

Washington State shows an electric vehicle market that is technologically
mature but commercially close to a monopoly. The challenge for the coming
years is no longer battery range, but diversifying the offer, so as to break
the current concentration of *market share* and widen access through new
manufacturers and new models eligible for the government incentives (*CAFV*).